# Logistic regression — reproducibility supplement

**Provenance:** the original 03 notebook was an empty file. Its historical CSVs remain unchanged. This notebook displays a separate post-hoc supplement, not recovered original training or an independent unseen-test experiment.

This notebook reads frozen artifacts only. See README for separate retraining commands.

In [1]:
from pathlib import Path
import json
from IPython.display import display
from integration.audit import read_csv

ROOT = Path.cwd()
assert (ROOT / "data/splits/test.csv").exists(), "Run from repository root"
OUT = ROOT / "results" / 'logistic_supplement'
protocol = json.loads((OUT / "protocol_frozen.json").read_text())
print(protocol.get("Evidence_status", "Historical frozen experiment; see integration audit for chronology"))
display(protocol["Selected_parameters"])

Post-hoc reproducibility supplement after historic test results existed


{'C': 0.1, 'class_weight': None}

In [2]:
display(read_csv(OUT / "cv_results.csv").sort_values("Mean_CV_AP", ascending=False).head())
display(read_csv(OUT / "test_metrics.csv")[["Model","Threshold","AP","ROC-AUC","Precision","Recall","F1","Accuracy","TN","FP","FN","TP","Cost_5","Alert_rate"]])

,Candidate,Parameters,Mean_CV_AP,SD_CV_AP,Mean_fit_AP,Mean_CV_ROC_AUC,Mean_fit_seconds,Fold_0_AP,Fold_1_AP,Fold_2_AP,Fold_3_AP,Fold_4_AP
5,5,"{""C"": 0.1, ""class_weight"": null}",0.511332,0.014929,0.513318,0.728978,0.034639,0.508403,0.507913,0.524555,0.526235,0.489554
10,10,"{""C"": 100.0, ""class_weight"": null}",0.511086,0.014846,0.513486,0.729072,0.022956,0.508554,0.507358,0.524409,0.525705,0.489403
0,0,"{""C"": 1.0, ""class_weight"": null}",0.511082,0.014828,0.513455,0.729025,0.027796,0.508519,0.507470,0.524353,0.525681,0.489387
8,8,"{""C"": 10.0, ""class_weight"": null}",0.511063,0.014837,0.513474,0.729050,0.024841,0.508563,0.507267,0.524353,0.525715,0.489416
3,3,"{""C"": 0.01, ""class_weight"": null}",0.510879,0.014310,0.511994,0.728225,0.022167,0.506843,0.507977,0.523243,0.525868,0.490463


,Model,Threshold,AP,ROC-AUC,Precision,Recall,F1,Accuracy,TN,FP,FN,TP,Cost_5,Alert_rate
0,Baseline LR supplement,0.500000,0.496998,0.711133,0.697531,0.255463,0.373966,0.810833,4526,147,988,339,5087,0.081000
1,Tuned LR supplement,0.500000,0.496749,0.710679,0.698545,0.253203,0.371681,0.810667,4528,145,991,336,5100,0.080167
2,Tuned LR supplement / cost ratio 1,0.445354,0.496749,0.710679,0.664157,0.332329,0.442993,0.815167,4450,223,886,441,4653,0.110667
3,Tuned LR supplement / cost ratio 3,0.265115,0.496749,0.710679,0.491661,0.510927,0.501109,0.775000,3972,701,649,678,3946,0.229833
4,Tuned LR supplement / cost ratio 5,0.234316,0.496749,0.710679,0.401018,0.593821,0.478736,0.714000,3496,1177,539,788,3872,0.327500
5,Tuned LR supplement / cost ratio 10,0.003941,0.496749,0.710679,0.221388,1.000000,0.362519,0.222167,6,4667,0,1327,4667,0.999000
6,Always negative,1.000000,0.221167,0.500000,0.000000,0.000000,0.000000,0.778833,4673,0,1327,0,6635,0.000000
7,Always positive,0.000000,0.221167,0.500000,0.221167,1.000000,0.362222,0.221167,0,4673,0,1327,4673,1.000000


In [3]:
display(read_csv(OUT / "bootstrap_intervals.csv"))
display(read_csv(OUT / "validation_importance.csv").head(10))

,Quantity,Estimate,CI_low,CI_high,Bootstrap_replicates
0,Tuned AP,0.496749,0.471423,0.524646,1000
1,Tuned ROC-AUC,0.710679,0.692342,0.726480,1000
2,AP difference: tuned - baseline,-0.000249,-0.000637,0.000121,1000
3,ROC-AUC difference: tuned - baseline,-0.000454,-0.000809,-0.000060,1000
4,Cost/1000 difference: tuned r=5 threshold - tu...,-204.666667,-233.675000,-175.487500,1000


,Feature,Mean_AP_decrease,SD_AP_decrease
0,PAY_0,0.212858,0.003632
1,BILL_AMT1,0.019258,0.003088
2,PAY_3,0.009021,0.001179
3,LIMIT_BAL,0.006569,0.002682
4,PAY_2,0.004975,0.001266
5,MARRIAGE,0.004780,0.000513
6,BILL_AMT3,0.004638,0.000775
7,BILL_AMT4,0.004346,0.001074
8,EDUCATION,0.003775,0.001195
9,BILL_AMT5,0.003519,0.000773


AP is average_precision_score, not trapezoidal PR-AUC. Bootstrap intervals condition on fixed predictions, model and thresholds; no retraining or threshold-selection uncertainty is included. Importance is associative, not causal.